# Attempt at taking the model from encoder and using it.



#Chritopher may have to change direction for when they are to use le model.

In [15]:
import pandas as pd
import os
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as T

In [16]:
# Christophers old Original 

class Encoder(nn.Module):
    def __init__(self, encoded_space_dim):
        super().__init__()

        self.encoder_cnn = nn.Sequential(
            nn.Conv2d(3, 8, 3, stride=2, padding=0),
            nn.ReLU(True),
            nn.Conv2d(8, 16, 3, stride=2, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(True),
            nn.Conv2d(16, 32, 3, stride=2, padding=0),
            nn.ReLU(True)
        )

        self.flatten = nn.Flatten(start_dim=1)

        self.encoder_lin = nn.Sequential(
            nn.Linear(11*9*32, 128),
            nn.ReLU(True),
            nn.Linear(128, encoded_space_dim)
        )

    def forward(self, x):
        x = self.encoder_cnn(x)
        x = self.flatten(x)
        x = self.encoder_lin(x)
        return x
        

In [17]:
model_dim  = 91
model = Encoder(model_dim)


state_dict = torch.load("padded_data_encoder", map_location="cpu") #dem model containen them weighen
model.load_state_dict(state_dict)
model.eval()

Encoder(
  (encoder_cnn): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
    (1): ReLU(inplace=True)
    (2): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
    (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): ReLU(inplace=True)
    (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
    (6): ReLU(inplace=True)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (encoder_lin): Sequential(
    (0): Linear(in_features=3168, out_features=128, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=128, out_features=91, bias=True)
  )
)

In [18]:
#Loading of directions
# --- load CSV ---
df_full = pd.read_csv(
    "../../data/grand_scraper_folder/unique_scrandle_pictures_FULL.csv"
)

base_folder = "../../data/scrandle_padded_low_res_data"
transform = T.ToTensor()

In [19]:
#Extraction 

images = [] #liste for le images

# --- extract first occurrence and load image ---
for occ in df_full["occurrences"]:
    
    first = occ.split("|")[0].strip()          # e.g. "2025-07-09:8_right"
    date, name = first.split(":")              # split
    
    path = os.path.join(
        base_folder,
        date,
        name + ".webp"
    )
    
    img = Image.open(path).convert("RGB")
    img = transform(img)
    
    images.append(img)

In [21]:
# --- stack ---
tensor_data = torch.stack(images) #Straking of the images

# --- encode ---
with torch.no_grad():
    encoded = model(tensor_data) #Using the model, which we

# --- dataframe ---
df_encoded = pd.DataFrame(encoded.numpy())

# --- save ---
df_encoded.to_csv("../../UlrikSteenAndersen/kloster_ring/encoded_unique.csv", index=False) #PLEASE USE ANOTHER NAME OR DIRECTION IF YOU WANT